# 05 — The teacher labels the unlabelled corpus

Step 5 of the protocol (D105; roadmap §B, steps 2-3). The teacher transcribes every clip of the
unlabelled corpus that `04_PreDistill.ipynb` cut into `DATASET_REPO/distill/` (D101), and its
labels are filtered before any student trains on them. Never the paid routes: the teacher on this
GPU is the only model.

**Teacher.** The model `03e_Flex_Ship.ipynb` froze: `FLEX_REPO/teacher.json` names it, and this
notebook reads that file rather than naming a model of its own. Greedy decoding, with per-token
log-probs kept. There is no loop retry: the filter drops every clip whose greedy output loops, so
retrying would spend GPU on labels that are thrown away. The labels go to a folder named after the
teacher, so another teacher's labels never mix with these.

**The filter, cheapest rule first** (`distill.filter_pseudo`): drop a clip overlapped for more
than `MAX_OVERLAP_SHARE` of its length (crosstalk is where the teacher is weakest; 04 measured it);
a clip whose output loops; an empty transcript; a rate outside the tokens per second train's
labels span, in the teacher's own tokens; then the least confident `DROP_FRACTION` of what is
left, by mean log-prob.

**Choose `MAX_OVERLAP_SHARE` from 04's table**, which prices each threshold in hours per channel.
It is `None` until then, which drops nothing for overlap. Every clip is decoded whatever the
threshold (about one GPU-hour per 100 h), so changing it later reruns only the filter cell.

**The report** splits what was kept by channel, by overlap bucket and by the share of English
words: every student fell furthest behind Flex on pure Nepali, and a threshold that empties the
round-table channels leaves a corpus of few voices.

**Small blast radius.** Labels go to `DATASET_REPO/<LABELS>/shards/` every `SHARD_CLIPS` clips, and
a rerun skips every clip already in a shard, so a lost runtime costs at most one shard.

**Smoke run first.** `SMOKE = True` reads `teacher-smoke.json`, decodes the first 500 clips and
writes under `distill/pseudo-smoke/`. The log-prob masking and the memory that `output_scores`
takes have not run on a GPU until it has.

## Config

In [ ]:
DATASET_REPO = "Sagyam/nepanglish-asr"     # 04_PreDistill.ipynb wrote the corpus to its distill/
PREFIX = "distill"
FLEX_REPO = "Sagyam/nepanglish-asr-flex-ft"   # its teacher.json names the model (03e_Flex_Ship.ipynb)
LANG, MODE = "ne", "mixed"
SMOKE = False              # True: teacher-smoke.json, the first 500 clips, labels under pseudo-smoke/
LIMIT = None               # decode only the first N clips; None for the whole corpus
SHARD_CLIPS = 2000         # clips per uploaded shard: a lost runtime loses at most one
DECODE_BUDGET_S, DECODE_ITEMS = 1200.0, 64
PAD_TO_S = 1.0
MAX_NEW_TOKENS = 300       # as the decoder gold is scored with
MAX_OVERLAP_SHARE = None   # drop a clip overlapped for more than this share; None drops none. From 04's table
DROP_FRACTION = 0.15       # the least confident share dropped (roadmap: 10-20%)

## Setup

In [ ]:
%pip install -q rapidfuzz
import json
import os
import sys
from pathlib import Path

# Before torch touches CUDA: lets the allocator grow segments instead of fragmenting when batch
# shapes vary (a fresh kernel only).
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

IN_COLAB = "google.colab" in sys.modules
# Secrets only work from a cell run in the Colab UI. When cells are driven from outside (the Colab
# MCP), run this cell by hand once. The token is then kept in the hub's own token file on the VM
# (where `hf auth login` puts it), so a kernel restart needs no click; the file goes when the
# runtime is deleted.
if IN_COLAB:
    from google.colab import userdata

    TOKEN_FILE = Path.home() / ".cache" / "huggingface" / "token"
    if not os.environ.get("HF_TOKEN"):
        os.environ["HF_TOKEN"] = (TOKEN_FILE.read_text().strip() if TOKEN_FILE.exists()
                                  else userdata.get("HF_TOKEN"))
    if not TOKEN_FILE.exists():
        TOKEN_FILE.parent.mkdir(parents=True, exist_ok=True)
        TOKEN_FILE.write_text(os.environ["HF_TOKEN"])
        TOKEN_FILE.chmod(0o600)
FT = Path("/content/ft") if IN_COLAB else Path.cwd() / ".cache-ft"
FT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(FT))
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
%%writefile /content/ft/ftkit.py
"""Shared fine-tuning kit for the Nepanglish ASR notebooks (Flex, 03a-03e; the students, 06).

Everything that is not model-specific: the dataset held in RAM, duration-bucketed batches, the
harness scorer, a GPU utilisation monitor, a batch-size probe and one training loop. Each notebook
writes this file out, so it imports exactly the same code as the main kernel. Keep it importable
on Python 3.10+ with numpy 1.x or 2.x.

How the GPU is kept busy:
  * audio is decoded once into RAM as int16; a clip is a slice, so no disk I/O in the loop;
  * batches are built by duration and padded to whole seconds, so there is little padding and
    only ~20 distinct shapes (cudnn.benchmark can then pick kernels once per shape);
  * DataLoader workers collate and pin the next batches while the GPU runs the current one;
  * a probe finds the largest micro-batch that survives forward+backward at the longest clip,
    with the optimizer state already allocated, and training uses a fixed fraction of it;
  * bf16 autocast, TF32 matmuls and fused AdamW; gradient accumulation reaches the effective
    batch; utilisation, throughput and padding waste are logged, not assumed.
"""

from __future__ import annotations

import glob
import json
import math
import random
import re
import shutil
import subprocess
import sys
import threading
import time
from collections import Counter
from collections.abc import Callable, Iterator, Sequence
from contextlib import contextmanager
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import numpy as np
import soundfile as sf
import torch

REPO = "Sagyam/nepanglish-asr"
SR = 16_000


# --- data --------------------------------------------------------------------------------------


def download_dataset(
    local: str | None = None, episodes: Sequence[str] = (), analytics: bool = False
) -> Path:
    """The HF dataset snapshot, or a local copy in the same layout.

    Labels, manifests and the harness's scorer always come; audio only for the recordings named
    in `episodes` (whole FLACs, 8 GiB for every split), and the analytics export, which carries
    the speaker turns, only when asked. Call it once without episodes to read the splits, then
    again with the episodes the notebook decodes or trains on: both calls return the same folder.
    """
    if local:
        return Path(local)
    from huggingface_hub import snapshot_download

    patterns = ["training/training.jsonl", "training/manifest.json", "gold/*", "harness/*"]
    if analytics:
        patterns.append("analytics/*")
    patterns += [f"training/episodes/{glob.escape(e)}.flac" for e in sorted(set(episodes))]
    return Path(snapshot_download(REPO, repo_type="dataset", allow_patterns=patterns))


def load_splits(data: Path) -> dict[str, list[dict]]:
    """train / val from the training export, gold from the gold export; asserts disjointness."""
    rows = [
        json.loads(line) for line in (data / "training" / "training.jsonl").open(encoding="utf-8")
    ]
    gold = [json.loads(line) for line in (data / "gold" / "gold.jsonl").open(encoding="utf-8")]
    splits = {
        "train": [r for r in rows if r["split"] == "train"],
        "val": [r for r in rows if r["split"] == "val"],
        "gold": gold,
    }
    trained = {r["segment_id"] for r in rows}
    assert not trained & {r["segment_id"] for r in gold}, "a gold clip is in the training export"
    return splits


def attach_speaker_turns(data: Path, rows: Sequence[dict]) -> int:
    """Copy each row's diarized turns, with their linked voices (D78, D87), from the analytics
    export: the training export does not carry them. Returns how many rows have turns."""
    want = {r["segment_id"]: r for r in rows}
    with (data / "analytics" / "analytics.jsonl").open(encoding="utf-8") as fh:
        for line in fh:
            a = json.loads(line)
            if a["segment_id"] in want:
                want[a["segment_id"]]["speaker_turns"] = a.get("speaker_turns")
    return sum(bool(r.get("speaker_turns")) for r in rows)


def duration(row: dict) -> float:
    return row["end_time"] - row["start_time"]


class AudioStore:
    """Every episode decoded once into RAM as int16; a clip is a slice of it (a view, no copy).

    ``folder`` is where the whole recordings are, under ``data``: the labelled export's
    ``training/episodes`` by default, or the distillation corpus's ``distill/episodes`` (D101)."""

    def __init__(self, data: Path, episode_ids: Sequence[str], folder: str = "training/episodes"):
        self.audio: dict[str, np.ndarray] = {}
        for ep in sorted(set(episode_ids)):
            audio, sr = sf.read(data / folder / f"{ep}.flac", dtype="int16")
            assert sr == SR and audio.ndim == 1, (ep, sr, audio.shape)
            self.audio[ep] = audio

    def clip(self, row: dict) -> np.ndarray:
        if "audio" in row:  # a public-benchmark clip carries its own audio (evalkit)
            return row["audio"]
        audio = self.audio[row["episode_id"]]
        return audio[round(row["start_time"] * SR) : round(row["end_time"] * SR)]

    def clip_f32(self, row: dict) -> torch.Tensor:
        return torch.from_numpy(self.clip(row).astype(np.float32) / 32768.0)

    @property
    def gib(self) -> float:
        return sum(a.nbytes for a in self.audio.values()) / 2**30


def bucket_batches(
    rows: Sequence[dict],
    *,
    budget_s: float,
    max_items: int,
    pad_to_s: float,
    shuffle: bool,
    seed: int = 0,
) -> list[list[int]]:
    """Index batches whose padded size (items x longest clip, rounded up to `pad_to_s`) fits
    `budget_s`. Sorting by duration keeps padding low; shuffling reorders whole batches and
    jitters lengths slightly so the same clips do not always share a batch."""
    rng = random.Random(seed)
    jitter = [rng.uniform(-0.3, 0.3) if shuffle else 0.0 for _ in rows]
    order = sorted(range(len(rows)), key=lambda i: duration(rows[i]) + jitter[i])
    batches: list[list[int]] = []
    cur: list[int] = []
    longest = 0.0
    for i in order:
        d = math.ceil(duration(rows[i]) / pad_to_s) * pad_to_s
        if cur and ((len(cur) + 1) * max(longest, d) > budget_s or len(cur) >= max_items):
            batches.append(cur)
            cur, longest = [], 0.0
        cur.append(i)
        longest = max(longest, d)
    if cur:
        batches.append(cur)
    if shuffle:
        rng.shuffle(batches)
    return batches


def pad_len(samples: int, pad_to_s: float) -> int:
    step = int(pad_to_s * SR)
    return math.ceil(samples / step) * step


class _Batches(torch.utils.data.Dataset):
    def __init__(self, rows, batches, collate):
        self.rows, self.batches, self.collate = rows, batches, collate

    def __len__(self):
        return len(self.batches)

    def __getitem__(self, i):
        return self.collate([self.rows[j] for j in self.batches[i]])


def loader(rows, batches, collate, workers: int) -> torch.utils.data.DataLoader:
    return torch.utils.data.DataLoader(
        _Batches(rows, batches, collate),
        batch_size=None,
        shuffle=False,
        num_workers=workers,
        pin_memory=True,
        prefetch_factor=4 if workers else None,
        persistent_workers=False,
    )


# --- scoring -----------------------------------------------------------------------------------


def is_loop(text: str) -> bool:
    """A 3-word sequence repeated 5 or more times: the decoder is stuck, not transcribing."""
    toks = text.split()
    top = Counter(zip(toks, toks[1:], toks[2:], strict=False)).most_common(1)
    return bool(top) and top[0][1] >= 5


class RetryLoops:
    """Wraps a batch `transcribe`: a clip whose first decode loops is decoded again, alone, by
    `retry` (anti-repetition settings); every other clip keeps its first decode untouched.
    `log` keeps (segment_id, first, retried), so the effect is measurable within one run."""

    def __init__(self, transcribe: Callable[[list[dict]], list[str]], retry: Callable[[dict], str]):
        self.transcribe, self.retry = transcribe, retry
        self.log: list[tuple[str, str, str]] = []

    def __call__(self, rows: list[dict]) -> list[str]:
        texts = self.transcribe(rows)
        for i, text in enumerate(texts):
            if is_loop(text):
                texts[i] = self.retry(rows[i])
                self.log.append((rows[i]["segment_id"], text, texts[i]))
        return texts


def harness_scorer(data: Path, work: Path) -> Callable[[Sequence[str], Sequence[str]], dict]:
    """fold.py from the dataset's harness/, imported from a real copy: the HF cache stores files as
    symlinks into its blob store, and normalize.py finds its config via its resolved path.

    Two layouts are accepted: the repo's (backend/app/services/, config/), and the flat one the
    2026-09-21 export uploaded (fold.py, normalize.py, normalization.yaml side by side), which
    is laid back out here because normalize.py looks for ../../../config/."""
    src, dst = data / "harness", work / "harness"
    if not dst.exists():
        if (src / "fold.py").exists():
            (dst / "backend" / "app" / "services").mkdir(parents=True)
            (dst / "config").mkdir()
            for name in ("fold.py", "normalize.py"):
                shutil.copyfile(src / name, dst / "backend" / "app" / "services" / name)
            shutil.copyfile(src / "normalization.yaml", dst / "config" / "normalization.yaml")
        else:
            shutil.copytree(src, dst)
    for name in [m for m in sys.modules if m == "app" or m.startswith("app.")]:
        del sys.modules[name]
    sys.path.insert(0, str(dst / "backend"))
    from app.services.fold import fold_tokens, fold_version, word_errors
    from rapidfuzz.distance import Levenshtein

    dev = re.compile(r"[ऀ-ॿ]")

    def chars(text: str) -> str:
        return " ".join(t if dev.search(t) else t.lower() for t in fold_tokens(text))

    def per_clip(refs: Sequence[str], hyps: Sequence[str]) -> list[dict]:
        """Every count a score needs, per clip, from one alignment each: `summarize` adds any
        subset of them up, so scoring per class or pairing against another system aligns
        nothing again. errors = sub + del + ins (folded)."""
        out = []
        for ref, hyp in zip(refs, hyps, strict=True):
            folded = word_errors(ref, hyp)
            raw = word_errors(ref, hyp, folded=False)
            ref_chars, hyp_chars = chars(ref), chars(hyp)
            out.append(
                {
                    "words": len(fold_tokens(ref)),
                    "errors": folded.errors,
                    "sub": folded.substitutions,
                    "del": folded.deletions,
                    "ins": folded.insertions,
                    "raw_words": raw.ref_words,
                    "raw_errors": raw.errors,
                    "chars": len(ref_chars),
                    "char_errors": Levenshtein.distance(ref_chars, hyp_chars),
                    "loop": int(is_loop(hyp)),
                }
            )
        return out

    def summarize(clips: Sequence[dict]) -> dict:
        """The score of a set of clips from their `per_clip` counts."""
        t = {k: sum(c[k] for c in clips) for k in clips[0]} if clips else Counter()
        words = max(t["words"], 1)
        folded = t["sub"] + t["del"] + t["ins"]
        assert folded == t["errors"], "S + D + I must add up to the folded errors"
        return {
            "wer": 100 * t["errors"] / words,
            "raw_wer": 100 * t["raw_errors"] / max(t["raw_words"], 1),
            "cer": 100 * t["char_errors"] / max(t["chars"], 1),
            # folded, per 100 reference words. In crosstalk a label keeps what was audible, which is
            # not always both voices, so a model that writes the other one can be charged insertions
            "sub": 100 * t["sub"] / words,
            "del": 100 * t["del"] / words,
            "ins": 100 * t["ins"] / words,
            "loops": t["loop"],
            "clips": len(clips),
        }

    def score(refs: Sequence[str], hyps: Sequence[str]) -> dict:
        return summarize(per_clip(refs, hyps))

    score.per_clip = per_clip
    score.summarize = summarize
    score.fold_version = fold_version()  # for the model card: which fold the numbers are in
    return score


def transcribe_rows(
    rows: Sequence[dict],
    transcribe: Callable[[list[dict]], list[str]],
    *,
    budget_s: float,
    max_items: int,
    pad_to_s: float,
) -> tuple[list[str], list[float]]:
    """Run `transcribe` over duration-bucketed batches; returns texts and per-clip compute seconds
    in the original row order."""
    texts: list[str] = [""] * len(rows)
    compute = [0.0] * len(rows)
    for batch in bucket_batches(
        rows, budget_s=budget_s, max_items=max_items, pad_to_s=pad_to_s, shuffle=False
    ):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        out = transcribe([rows[i] for i in batch])
        torch.cuda.synchronize()
        took = time.perf_counter() - t0
        audio_s = sum(duration(rows[i]) for i in batch)
        for i, text in zip(batch, out, strict=True):
            texts[i] = text.strip()
            compute[i] = took * duration(rows[i]) / audio_s
    return texts, compute


def write_hyps(
    path: Path, rows: Sequence[dict], texts: Sequence[str], compute: Sequence[float]
) -> None:
    """The hypothesis-cache format the bake-off comparisons used, so the result can join them."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as fh:
        for r, text, c in zip(rows, texts, compute, strict=True):
            fh.write(
                json.dumps(
                    {"segment_id": r["segment_id"], "text": text, "compute_s": c},
                    ensure_ascii=False,
                )
                + "\n"
            )


# --- GPU ---------------------------------------------------------------------------------------


def fast_cuda() -> None:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True  # safe: shapes are padded to whole seconds
    torch.set_float32_matmul_precision("high")


class GpuMonitor:
    """Samples nvidia-smi in a background thread; `window()` returns the mean utilisation and the
    peak memory since the previous call."""

    def __init__(self, every: float = 1.0):
        self.every, self._util, self._mem = every, [], []
        self._stop = threading.Event()
        self._thread = threading.Thread(target=self._run, daemon=True)

    def _run(self):
        while not self._stop.is_set():
            try:
                out = subprocess.run(
                    [
                        "nvidia-smi",
                        "--query-gpu=utilization.gpu,memory.used",
                        "--format=csv,noheader,nounits",
                    ],
                    capture_output=True,
                    text=True,
                    timeout=5,
                )
                util, mem = (float(x) for x in out.stdout.strip().splitlines()[0].split(","))
                self._util.append(util)
                self._mem.append(mem)
            except Exception:
                pass
            self._stop.wait(self.every)

    def start(self) -> GpuMonitor:
        self._thread.start()
        return self

    def stop(self) -> None:
        self._stop.set()

    def window(self) -> dict:
        util, mem = self._util, self._mem
        self._util, self._mem = [], []
        return {
            "gpu_util": float(np.mean(util)) if util else float("nan"),
            "gpu_mem_gib": max(mem) / 1024 if mem else float("nan"),
        }


def init_optimizer_state(model: torch.nn.Module, optimizer: torch.optim.Optimizer) -> None:
    """Allocate AdamW's moment buffers before probing, without moving any weight: with zero
    gradients and zero weight decay an AdamW step is a no-op, but it creates the state."""
    decay = [g["weight_decay"] for g in optimizer.param_groups]
    for g in optimizer.param_groups:
        g["weight_decay"] = 0.0
    for p in model.parameters():
        if p.requires_grad:
            p.grad = torch.zeros_like(p)
    optimizer.step()
    optimizer.zero_grad(set_to_none=True)
    for g, d in zip(optimizer.param_groups, decay, strict=True):
        g["weight_decay"] = d
    for state in optimizer.state.values():
        if "step" in state:
            state["step"].zero_()


def probe_max_items(
    step: Callable[[int], None], lo: int, hi: int, params: Sequence[torch.nn.Parameter]
) -> int:
    """The largest n in [lo, hi] for which `step(n)` (one forward+backward at the worst case)
    fits in memory. `step` must not touch the gradients.

    The gradient buffer stays allocated throughout, as it does in training from the second
    micro-batch of every accumulated step: freeing it between probes measured the activations
    without it and picked a batch that ran out of memory once training accumulated."""
    for p in params:
        p.grad = torch.zeros_like(p)
    best = 0
    while lo <= hi:
        mid = (lo + hi) // 2
        try:
            step(mid)
            torch.cuda.synchronize()
            best, lo = mid, mid + 1
        except torch.cuda.OutOfMemoryError:
            hi = mid - 1
        for p in params:
            p.grad.zero_()
        torch.cuda.empty_cache()
    for p in params:
        p.grad = None
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    return best


# --- training ----------------------------------------------------------------------------------


@dataclass
class TrainConfig:
    name: str
    out: str
    epochs: int = 10
    lr: float = 1e-5
    schedule: str = "linear"  # linear warmup+decay, or "tristage" (fairseq2's default)
    warmup_frac: float = 0.1
    effective_s: float = 720.0  # audio seconds per optimizer step, reached by accumulation
    weight_decay: float = 0.0
    clip_norm: float = 1.0
    patience: int = 3  # evaluations without a val-WER improvement before stopping
    # WER points an evaluation must gain on the last counted one to reset `patience`. 0 counts
    # any gain, so a crawl of 0.2 points an epoch never stops. The best weights are kept either way.
    min_delta: float = 0.0
    seed: int = 0
    log_every: int = 10
    workers: int = 6


def lr_factor(step: int, total: int, cfg: TrainConfig) -> float:
    warm = max(1, int(cfg.warmup_frac * total))
    if step < warm:
        return (step + 1) / warm
    if cfg.schedule == "tristage":  # 10% warmup, 40% hold, 50% exponential decay to 5%
        hold_end = int(0.5 * total)
        if step < hold_end:
            return 1.0
        frac = (step - hold_end) / max(1, total - hold_end)
        return math.exp(math.log(0.05) * frac)
    return max(0.0, (total - step) / max(1, total - warm))


def group_steps(rows, batches, effective_s: float) -> list[list[list[int]]]:
    """Micro-batches grouped into optimizer steps of about `effective_s` seconds of real audio."""
    steps, cur, acc = [], [], 0.0
    for b in batches:
        cur.append(b)
        acc += sum(duration(rows[i]) for i in b)
        if acc >= effective_s:
            steps.append(cur)
            cur, acc = [], 0.0
    if cur:
        steps.append(cur)
    return steps


@contextmanager
def timed(label: str) -> Iterator[None]:
    """Print `label...` before a step and `label: done in N s` after it, so a step that stalls
    shows which one it is instead of leaving a cell silent."""
    print(f"{label}...", flush=True)
    t0 = time.perf_counter()
    yield
    print(f"{label}: done in {time.perf_counter() - t0:.0f} s", flush=True)


def sid(m: dict) -> str:
    """A score's error split, per 100 reference words: S + D + I is its folded WER. Read all three
    on crosstalk: the labels drop the other voice, so hearing it costs insertions."""
    return f"S {m['sub']:.2f}  D {m['del']:.2f}  I {m['ins']:.2f}"


def retry_note(val: dict) -> str:
    """For an `evaluate` that wraps its decode in RetryLoops and reports the greedy score too."""
    if "retried" not in val:
        return ""
    return f"  (greedy WER {val['greedy_wer']:.2f}, {val['retried']} retried)"


def speed_check(
    model: torch.nn.Module,
    *,
    rows: Sequence[dict],
    batches: list[list[int]],
    collate: Callable[[list[dict]], Any],
    loss_fn: Callable[[torch.nn.Module, Any], tuple[torch.Tensor, int]],
    evaluate: Callable[[torch.nn.Module], dict],
    val_rows: Sequence[dict],
    gold_rows: Sequence[dict],
    epochs: int,
    monitor: GpuMonitor,
    workers: int = 6,
    n_micro: int = 10,
) -> dict:
    """Time the run before committing to it: forward+backward on `n_micro` real training
    micro-batches (no optimizer step, so no weight moves), then one full val pass, which is also
    the val WER before training. Projects the wall time of every epoch and the gold pass.

    The micro-batches run twice and only the second pass is timed, so cudnn's per-shape kernel
    search and worker start-up are excluded. Gradients stay allocated between micro-batches, as
    they do under accumulation, so the memory reading is training's. BatchNorm running
    statistics are restored after."""
    bns = [m for m in model.modules() if isinstance(m, torch.nn.modules.batchnorm._BatchNorm)]
    saved = [{k: v.clone() for k, v in m.state_dict().items()} for m in bns]
    sample = batches[:n_micro]
    it = iter(loader(rows, sample + sample, collate, workers))

    def run() -> tuple[float, float, int]:
        audio = padded = 0.0
        clips = 0
        for _ in sample:
            b = next(it)
            with torch.autocast("cuda", dtype=torch.bfloat16):
                loss, _ = loss_fn(model, b)
            loss.backward()
            model.zero_grad(set_to_none=False)
            audio, padded = audio + b["seconds"], padded + b["padded_seconds"]
            clips += len(b["lens"]) if "lens" in b else b["wav"].shape[0]
        torch.cuda.synchronize()
        return audio, padded, clips

    model.train()
    run()
    monitor.window()
    t0 = time.perf_counter()
    audio, padded, clips = run()
    train_dt = time.perf_counter() - t0
    gpu = monitor.window()
    model.zero_grad(set_to_none=True)
    for m, s in zip(bns, saved, strict=True):
        m.load_state_dict(s)

    model.eval()
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        val = evaluate(model)
    torch.cuda.synchronize()
    val_dt = time.perf_counter() - t0
    model.train()

    secs = lambda rs: sum(duration(r) for r in rs)  # noqa: E731
    epoch_s = secs(rows) / (audio / train_dt)
    gold_s = val_dt * secs(gold_rows) / secs(val_rows)
    rec = {
        "train_x_realtime": audio / train_dt,
        "train_ms_per_clip": 1000 * train_dt / clips,
        "padding_waste": 1 - audio / padded,
        **gpu,
        "val_s": val_dt,
        "val_ms_per_clip": 1000 * val_dt / len(val_rows),
        **{f"val_before_{k}": v for k, v in val.items()},
        "epoch_min": epoch_s / 60,
        "projected_h": (epochs * (epoch_s + val_dt) + gold_s) / 3600,
    }
    print(
        f"train: {rec['train_x_realtime']:.0f}x realtime = "
        f"{rec['train_ms_per_clip']:.0f} ms per clip "
        f"(forward+backward, {clips} clips), pad waste {rec['padding_waste']:.0%}, "
        f"GPU {rec['gpu_util']:.0f}% util, {rec['gpu_mem_gib']:.1f} GiB\n"
        f"val:   {val_dt:.0f} s for {len(val_rows)} clips = "
        f"{rec['val_ms_per_clip']:.0f} ms per clip "
        f"(batched decode); WER before training {val['wer']:.2f} ({sid(val)}), loops {val['loops']}"
        f"{retry_note(val)}\n"
        f"projected: {rec['epoch_min']:.1f} min per epoch + {val_dt / 60:.1f} min val -> "
        f"at most {rec['projected_h']:.2f} h for {epochs} epochs and gold "
        "(early stopping can cut it)",
        flush=True,
    )
    return rec


class HubResume:
    """One run's resume point in a scratch model repo on the hub: `<path>/resume.pt`.

    `train` saves it after every epoch's evaluation and reads it when it starts, so a lost
    runtime costs the epoch in progress, not the run. It holds the weights as they are, the best
    weights when they differ, the counters and the history; not the optimizer's moments, which
    are twice the model's size and rebuild within a few steps. Every save replaces the last one,
    and the blobs it supersedes are deleted from the repo's storage, which is why this lives in a
    repo of its own: nothing else can be deleted by mistake. A failed save, load or clean-up is
    reported and never stops training."""

    NAME = "resume.pt"

    def __init__(self, api: Any, repo: str, path: str, local: Path):
        self.api, self.repo, self.path, self.local = api, repo, path, Path(local)

    @property
    def remote(self) -> str:
        return f"{self.path}/{self.NAME}"

    def load(self) -> dict | None:
        try:
            if not self.api.file_exists(self.repo, self.remote):
                return None
            from huggingface_hub import hf_hub_download

            with timed(f"resuming from {self.repo}/{self.remote}"):
                path = hf_hub_download(self.repo, self.remote, token=self.api.token)
                return torch.load(path, map_location="cpu", weights_only=True)
        except Exception as exc:  # a broken resume point means starting over, not failing
            print(
                f"no usable resume point ({type(exc).__name__}: {exc}); starting over", flush=True
            )
            return None

    def save(self, state: dict) -> None:
        try:
            self.local.mkdir(parents=True, exist_ok=True)
            torch.save(state, self.local / self.NAME)
            with timed(f"saving the resume point after epoch {state['epoch']}"):
                self.api.upload_file(
                    path_or_fileobj=str(self.local / self.NAME),
                    path_in_repo=self.remote,
                    repo_id=self.repo,
                    commit_message=f"{self.path}: after epoch {state['epoch']}",
                )
            self._purge(keep_newest=True)
        except Exception as exc:
            print(f"resume point not saved ({type(exc).__name__}: {exc})", flush=True)

    def clear(self) -> None:
        """Remove the resume point once the run's weights are safely uploaded elsewhere."""
        try:
            if self.api.file_exists(self.repo, self.remote):
                self.api.delete_file(
                    self.remote, self.repo, commit_message=f"{self.path}: run finished"
                )
            self._purge(keep_newest=False)
            shutil.rmtree(self.local, ignore_errors=True)
        except Exception as exc:
            print(f"resume point not cleared ({type(exc).__name__}: {exc})", flush=True)

    def _purge(self, keep_newest: bool) -> None:
        """Delete this run's superseded blobs for good: a replaced file stays in the repo's
        history, and its storage stays counted, until it is removed from LFS as well."""
        mine = sorted(
            (f for f in self.api.list_lfs_files(self.repo) if f.filename == self.remote),
            key=lambda f: f.pushed_at,
        )
        stale = mine[:-1] if keep_newest else mine
        if stale:
            self.api.permanently_delete_lfs_files(self.repo, stale)


def train(
    model: torch.nn.Module,
    *,
    cfg: TrainConfig,
    rows: Sequence[dict],
    make_batches: Callable[[int], list[list[int]]],
    collate: Callable[[list[dict]], Any],
    loss_fn: Callable[[torch.nn.Module, Any], tuple[torch.Tensor, int]],
    evaluate: Callable[[torch.nn.Module], dict],
    save_best: Callable[[torch.nn.Module], None],
    optimizer: torch.optim.Optimizer,
    monitor: GpuMonitor,
    resume: Any | None = None,
) -> dict:
    """Train with bf16 autocast and gradient accumulation; evaluate on val after every epoch;
    keep the best weights (by val WER) in CPU memory and hand them to `save_best`. A hand stop
    (KeyboardInterrupt) after the first evaluation ends training the same way.

    `loss_fn` returns a *summed* loss and the number of units it sums over (clips for CTC, tokens
    for cross-entropy); gradients are divided by the step's total units, so accumulated
    micro-batches of different sizes are weighted exactly. Count the units on the CPU (in
    `collate`): an `int()` of a GPU tensor stalls the host once per micro-batch.

    `resume` (a `HubResume`) makes the run restartable: its state is saved after every epoch and,
    when one is found at the start, training continues from the epoch after it, on the same
    batches and at the same point of the LR schedule. The optimizer's moments start again."""
    out = Path(cfg.out)
    out.mkdir(parents=True, exist_ok=True)
    torch.manual_seed(cfg.seed)
    plan = [group_steps(rows, make_batches(e), cfg.effective_s) for e in range(cfg.epochs)]
    total = sum(len(p) for p in plan)
    base_lrs = [g["lr"] for g in optimizer.param_groups]
    params = [p for p in model.parameters() if p.requires_grad]
    history, best, best_state, bad, step = [], float("inf"), None, 0, 0
    counted = float("inf")  # the val WER that last reset `bad`
    first_epoch, finished = 0, False
    state = resume.load() if resume is not None else None
    if state is not None:
        model.load_state_dict(state["model"])
        best_state = state["best_model"] or state["model"]
        history, best, counted = state["history"], state["best"], state["counted"]
        bad, step, first_epoch, finished = (
            state["bad"],
            state["step"],
            state["epoch"],
            state["done"],
        )
        print(
            f"{cfg.name}: resumed after epoch {first_epoch} (best val WER {best:.2f})", flush=True
        )
    print(
        f"{cfg.name}: {total} optimizer steps over {cfg.epochs} epochs "
        f"(~{cfg.effective_s / 60:.0f} min of audio each), peak lr {cfg.lr:g}, {cfg.schedule}"
    )
    stopped_by_hand = False
    try:
        for epoch in range(first_epoch, 0 if finished else cfg.epochs):
            model.train()
            flat = [b for s in plan[epoch] for b in s]
            sizes = [len(s) for s in plan[epoch]]
            it = iter(loader(rows, flat, collate, cfg.workers))
            t_log, audio_log, padded_log, loss_log, units_log = (
                time.perf_counter(),
                0.0,
                0.0,
                0.0,
                0,
            )
            monitor.window()
            for n_micro in sizes:
                for g, lr0 in zip(optimizer.param_groups, base_lrs, strict=True):
                    g["lr"] = lr0 * lr_factor(step, total, cfg)
                step_units = 0
                for _ in range(n_micro):
                    batch = next(it)
                    with torch.autocast("cuda", dtype=torch.bfloat16):
                        loss, units = loss_fn(model, batch)
                    loss.backward()
                    step_units += units
                    loss_log += loss.detach()  # stays on the GPU: no host sync per micro-batch
                    units_log += units
                    audio_log += batch["seconds"]
                    padded_log += batch["padded_seconds"]
                torch._foreach_div_(
                    [p.grad for p in params if p.grad is not None], max(step_units, 1)
                )
                gnorm = torch.nn.utils.clip_grad_norm_(params, cfg.clip_norm)
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                step += 1
                if step % cfg.log_every == 0:
                    torch.cuda.synchronize()
                    dt = time.perf_counter() - t_log
                    gpu = monitor.window()
                    rec = {
                        "step": step,
                        "epoch": epoch + 1,
                        "lr": optimizer.param_groups[0]["lr"],
                        "loss": float(loss_log) / max(units_log, 1),
                        "grad_norm": float(gnorm),
                        "audio_x_realtime": audio_log / dt,
                        "padding_waste": 1 - audio_log / max(padded_log, 1e-9),
                        "peak_alloc_gib": torch.cuda.max_memory_allocated() / 2**30,
                        **gpu,
                    }
                    history.append(rec)
                    print(
                        f"step {step:5d}/{total} ep {epoch + 1} lr {rec['lr']:.2e} "
                        f"loss {rec['loss']:.4f} | {rec['audio_x_realtime']:6.0f}x realtime, "
                        f"pad waste {rec['padding_waste']:.0%}, "
                        f"GPU {rec['gpu_util']:.0f}% util, {rec['gpu_mem_gib']:.1f} GiB",
                        flush=True,
                    )
                    t_log, audio_log, padded_log, loss_log, units_log = (
                        time.perf_counter(),
                        0.0,
                        0.0,
                        0.0,
                        0,
                    )
            model.eval()
            with torch.no_grad():
                val = evaluate(model)
            history.append(
                {"epoch": epoch + 1, "step": step, **{f"val_{k}": v for k, v in val.items()}}
            )
            improved = val["wer"] < best
            print(
                f"== epoch {epoch + 1}: val WER {val['wer']:.2f} ({sid(val)})  "
                f"CER {val['cer']:.2f}  raw WER {val['raw_wer']:.2f}  "
                f"loops {val['loops']}{retry_note(val)}" + ("  (best)" if improved else ""),
                flush=True,
            )
            (out / "history.json").write_text(json.dumps(history, indent=1))
            current = None
            if improved or resume is not None:  # a copy on the CPU: the run's only other one
                current = {
                    k: v.detach().to("cpu", copy=True) for k, v in model.state_dict().items()
                }
            if improved:
                best_state, best = current, val["wer"]
            if val["wer"] < counted - cfg.min_delta:  # at 0: the old rule, any strict gain
                counted, bad = val["wer"], 0
            else:
                bad += 1
            finished = bad >= cfg.patience
            if resume is not None:
                resume.save(
                    {
                        "epoch": epoch + 1,
                        "step": step,
                        "best": best,
                        "counted": counted,
                        "bad": bad,
                        "done": finished,
                        "history": history,
                        "model": current,
                        "best_model": None if improved else best_state,
                    }
                )
            del current
            if finished:
                print(
                    f"val WER gained less than {cfg.min_delta:g} points in {cfg.patience} "
                    "evaluations; stopping"
                )
                break
    except KeyboardInterrupt:
        # Colab's stop button: end training here and keep the best weights so far, so they are
        # saved and scored like a finished run's. Before any evaluation there are none to keep.
        if best_state is None:
            raise
        stopped_by_hand = True
        print(f"stopped by hand; keeping the best weights (val WER {best:.2f})", flush=True)
    monitor.window()
    with timed("saving the best weights"):
        if best_state is not None:
            model.load_state_dict(best_state)
        model.eval()
        save_best(model)
    (out / "config.json").write_text(json.dumps(asdict(cfg), indent=1))
    return {
        "best_val_wer": best,
        "steps": step,
        "history": history,
        "stopped_by_hand": stopped_by_hand,
    }

In [ ]:
%%writefile /content/ft/distill.py
"""Pure helpers for the distillation notebook (roadmap §B).

No torch and no NeMo, so the rules that decide what shapes the students' tokenizer and what their
scores are paired against are tested from backend/tests. Keep it importable on Python 3.10+.
"""

from __future__ import annotations

import re
import unicodedata
from collections.abc import Callable, Mapping, Sequence
from typing import Any


def tokenizer_texts(splits: Mapping[str, Sequence[dict]]) -> list[str]:
    """The labels the shared student tokenizer is trained on: train only.

    Val and gold text never shapes the vocabulary, so a word only they contain is scored as the
    student would meet it on new audio. Raises if a val or gold clip is in train."""
    held = {r["segment_id"] for name in ("val", "gold") for r in splits.get(name, ())}
    leaked = sorted(held & {r["segment_id"] for r in splits["train"]})
    if leaked:
        raise ValueError(f"held-out clips in train: {leaked[:5]}")
    return [r["text"] for r in splits["train"]]


def reference_texts(rows: Sequence[dict], by_id: Mapping[str, str]) -> list[str]:
    """Another system's transcripts in `rows`' order, for pairing clip by clip. Raises, naming
    them, if any row has none: a pairing over a subset would not be the same clips."""
    missing = [r["segment_id"] for r in rows if r["segment_id"] not in by_id]
    if missing:
        raise ValueError(f"{len(missing)} clip(s) have no reference transcript: {missing[:5]}")
    return [by_id[r["segment_id"]] for r in rows]


def by_class(
    rows: Sequence[dict],
    clips: Sequence[Any],
    summarize: Callable[[list[Any]], Any],
    keys: Sequence[str] | None = None,
) -> dict[str, dict[str, Any]]:
    """`summarize` per value of each clip class (D87), as {key: {value: summary}}.

    `clips` holds one per-clip count per row (`score.per_clip`), aligned once by the caller: a
    group's score is its clips' counts added up, so no clip is aligned again per key. `keys`
    defaults to every class key any row carries. A clip without a key (or with no classes) is left
    out of that key's groups, not counted under a made-up value."""
    if len(rows) != len(clips):
        raise ValueError("rows and clips must describe the same clips")
    if keys is None:
        keys = sorted({k for r in rows for k in (r.get("classes") or {})})
    out: dict[str, dict[str, Any]] = {}
    for key in keys:
        groups: dict[str, list[int]] = {}
        for i, r in enumerate(rows):
            value = (r.get("classes") or {}).get(key)
            if value is not None:
                groups.setdefault(str(value), []).append(i)
        if groups:
            out[key] = {v: summarize([clips[i] for i in idx]) for v, idx in sorted(groups.items())}
    return out


def filter_pseudo(
    rows: Sequence[dict],
    *,
    tokens_per_s: tuple[float, float],
    drop_fraction: float,
    max_overlap_share: float | None = None,
) -> tuple[list[dict], dict[str, Any]]:
    """Roadmap §B step 3: keep the teacher's labels worth training on, cheapest rule first.

    Each row carries ``segment_id``, ``text``, ``n_tokens`` (teacher tokens), ``duration`` (s),
    ``mean_logprob`` and ``looped`` (the greedy output loops, ``ftkit.is_loop``), and, once
    PreDistill has measured it, ``overlap_share``. Dropped, in order: a clip overlapped for more
    than ``max_overlap_share`` of its length (crosstalk is where the teacher is weakest; ``None``
    applies no such rule, and a clip never measured is kept and counted); a clip whose output
    loops; an empty transcript; a rate outside ``tokens_per_s`` (the range train's labels span, in
    the teacher's tokens); then the least confident ``drop_fraction`` of what is left, by mean
    log-prob. Returns the kept rows, in input order, and a report of what each rule dropped.
    """
    if not 0.0 <= drop_fraction < 1.0:
        raise ValueError(f"drop_fraction must be in [0, 1), not {drop_fraction}")
    if max_overlap_share is not None and not 0.0 <= max_overlap_share <= 1.0:
        raise ValueError(f"max_overlap_share must be in [0, 1], not {max_overlap_share}")
    lo, hi = tokens_per_s
    dropped = {"overlap": 0, "loop": 0, "empty": 0, "rate": 0, "confidence": 0}
    unmeasured = sum(r.get("overlap_share") is None for r in rows)
    left = []
    for r in rows:
        share = r.get("overlap_share")
        if max_overlap_share is not None and share is not None and share > max_overlap_share:
            dropped["overlap"] += 1
        elif r["looped"]:
            dropped["loop"] += 1
        elif not r["text"].strip():
            dropped["empty"] += 1
        elif not lo <= r["n_tokens"] / r["duration"] <= hi:
            dropped["rate"] += 1
        else:
            left.append(r)
    cut = None
    n_drop = int(len(left) * drop_fraction)
    if n_drop:
        least = sorted(left, key=lambda r: r["mean_logprob"])[:n_drop]
        cut = max(r["mean_logprob"] for r in least)
        gone = {r["segment_id"] for r in least}
        left = [r for r in left if r["segment_id"] not in gone]
        dropped["confidence"] = n_drop
    report = {
        "total": len(rows),
        "kept": len(left),
        "kept_hours": round(sum(r["duration"] for r in left) / 3600, 3),
        "dropped": dropped,
        "logprob_cut": cut,
        "tokens_per_s": [lo, hi],
        "drop_fraction": drop_fraction,
        "max_overlap_share": max_overlap_share,
        "overlap_unmeasured": unmeasured,
    }
    return left, report


def latin_share(text: str) -> float:
    """The share of a transcript's words written in Latin script: a code-mixing proxy that needs
    only the text, for pseudo-labels, which have no CMI class."""
    words = [w for w in text.split() if any(c.isalpha() for c in w)]
    if not words:
        return 0.0
    return sum(any("a" <= c.lower() <= "z" for c in w) for w in words) / len(words)


def mixing_bucket(share: float) -> str:
    """``latin_share`` in the corpus's CMI buckets (D87): 0, <15, 15-30, 30+ percent."""
    if share == 0:
        return "0"
    if share < 0.15:
        return "<15"
    return "15-30" if share < 0.30 else "30+"


# --- PreDistill: the owner's zip of recordings (D101) ------------------------------------------

_NUMBER = re.compile(r"_\d+$")


def _path_safe(text: str) -> str:
    """Letters and digits of any script, combining marks (Devanagari's vowel signs, which regex's
    word class misses), ``.``, ``-`` and ``_`` kept; every other run of characters becomes one
    ``_``."""
    kept = "".join(
        c if c.isalnum() or c in "._-" or unicodedata.category(c).startswith("M") else "\0"
        for c in text
    )
    return re.sub("\0+", "_", kept).strip("_")


def source_from_filename(name: str) -> tuple[str, str]:
    """A recording named ``<channel_name>_<NN>.mp3`` as ``(source_id, channel)``.

    The source id is the file's stem with every run of characters unsafe in a path turned into
    one ``_`` (letters of any script, digits, ``.``, ``-`` and ``_`` are kept); the channel is the
    id less its trailing ``_<number>``.
    """
    stem = name.replace("\\", "/").rsplit("/", 1)[-1].rsplit(".", 1)[0]
    source_id = _path_safe(stem)
    return source_id, _NUMBER.sub("", source_id) or source_id


def channel_blocked(channel: str, blocked: Sequence[str]) -> bool:
    """Whether a channel's name contains a blocked name, ignoring case and ``_`` against spaces."""
    name = channel.replace("_", " ").casefold()
    return any(b.strip().casefold() in name for b in blocked if b.strip())


def slice_rows(source_id: str, channel: str, slices: Sequence[tuple[float, float]]) -> list[dict]:
    """The manifest rows of one recording's VAD slices, named and timed as the labelled export's
    rows are, so ``ftkit.AudioStore`` cuts them from the whole recording."""
    return [
        {
            "segment_id": f"{source_id}_{i:05d}",
            "episode_id": source_id,
            "source_id": source_id,
            "channel": channel,
            "start_time": round(start, 3),
            "end_time": round(end, 3),
            "duration": round(end - start, 3),
        }
        for i, (start, end) in enumerate(slices)
    ]


# --- overlapped speech on the unlabelled corpus (D105) -------------------------------------------
#
# The harness measures overlap per episode (app/services/overlap.py) and classes a clip by the
# share of it that is overlapped (app/services/clip_classes.py). The second module imports
# SQLAlchemy and the models, so it cannot be fetched into Colab; its two pure functions are
# repeated here, and a test keeps them equal to the harness's.


def spans_within(
    spans: Sequence[Sequence[float]], start: float, end: float
) -> list[tuple[float, float]]:
    """A recording's overlap spans cut to one clip, clip-relative, as the labelled export's
    ``overlap_spans`` are. Empty when the clip holds none."""
    out: list[tuple[float, float]] = []
    for lo, hi in spans:
        a, b = max(lo, start), min(hi, end)
        if b > a:
            out.append((round(a - start, 3), round(b - start, 3)))
    return out


def overlap_share(spans: Sequence[Sequence[float]] | None, duration: float) -> float | None:
    """The fraction of a clip spent in crosstalk; ``None`` when it was never measured."""
    if spans is None:
        return None
    if duration <= 0:
        return 0.0
    return min(1.0, sum(max(0.0, end - start) for start, end in spans) / duration)


def overlap_bucket(share: float | None) -> str:
    """The overlap bucket of a clip's overlap share, the corpus's clip class (D87)."""
    if share is None:
        return "unmeasured"
    if share <= 0:
        return "none"
    if share < 0.05:
        return "0-5%"
    if share <= 0.15:
        return "5-15%"
    return ">15%"


def with_overlap(rows: Sequence[dict], spans: Sequence[Sequence[float]] | None) -> list[dict]:
    """One recording's clip rows with ``overlap_spans`` (clip-relative) and ``overlap_share``,
    from the recording's overlap spans. ``None`` spans (no detector) leave both ``None``: never
    measured, which is not the same as clean."""
    out = []
    for r in rows:
        mine = None if spans is None else spans_within(spans, r["start_time"], r["end_time"])
        share = overlap_share(mine, r["end_time"] - r["start_time"])
        out.append(
            {
                **r,
                "overlap_spans": None if mine is None else [list(s) for s in mine],
                "overlap_share": None if share is None else round(share, 4),
            }
        )
    return out


def overlap_loss(rows: Sequence[dict], thresholds: Sequence[float]) -> dict[str, dict[str, Any]]:
    """What a ``max_overlap_share`` would cost, per channel and for ``all``: the hours of speech,
    the hours never measured, and the hours dropped at each threshold (a clip is dropped when its
    share is *above* the threshold, as ``filter_pseudo`` drops it). The table the owner reads
    before choosing the threshold."""
    out: dict[str, dict[str, Any]] = {}
    for r in rows:
        hours = (r["end_time"] - r["start_time"]) / 3600
        share = r.get("overlap_share")
        for name in (r["channel"], "all"):
            c = out.setdefault(
                name,
                {
                    "hours": 0.0,
                    "unmeasured_hours": 0.0,
                    "dropped_hours": dict.fromkeys(thresholds, 0.0),
                },
            )
            c["hours"] += hours
            if share is None:
                c["unmeasured_hours"] += hours
                continue
            for threshold in thresholds:
                if share > threshold:
                    c["dropped_hours"][threshold] += hours
    return dict(sorted(out.items(), key=lambda kv: (kv[0] == "all", kv[0])))


# --- stage 2 of a student: pseudo-labels and human labels in one mixture (D106) ------------------


def mixture_weights(
    groups: Sequence[str | None], hours: Sequence[float], human_share: float
) -> list[float]:
    """Each clip's chance of being drawn, summing to 1.

    ``groups[i]`` is clip i's channel for a pseudo-labelled clip and ``None`` for a human-labelled
    one; ``hours[i]`` is its length. The human labels take ``human_share`` of the draws. The rest
    is split between channels in proportion to the *square root* of their hours (roadmap §B,
    2026-09-26: no channel is capped, and a 22 h channel weighs about 2x a 5 h one, not 4.4x).
    Inside a channel, and inside the human labels, a clip is drawn in proportion to its length.
    With no pseudo-labelled clip, or no human one, the other kind takes every draw.
    """
    if len(groups) != len(hours):
        raise ValueError("groups and hours must describe the same clips")
    if not 0.0 < human_share < 1.0:
        raise ValueError(f"human_share must be in (0, 1), not {human_share}")
    total: dict[str | None, float] = {}
    for g, h in zip(groups, hours, strict=True):
        if h <= 0:
            raise ValueError("every clip needs a positive length")
        total[g] = total.get(g, 0.0) + h
    channels = {g: h for g, h in total.items() if g is not None}
    roots = sum(h**0.5 for h in channels.values())
    share: dict[str | None, float] = {}
    if None in total:
        share[None] = human_share if channels else 1.0
    for g, h in channels.items():
        share[g] = (1.0 - human_share if None in total else 1.0) * h**0.5 / roots
    return [share[g] * h / total[g] for g, h in zip(groups, hours, strict=True)]


def draw_epoch(weights: Sequence[float], n: int, seed: int) -> list[int]:
    """The ``n`` clips of one epoch, as indices drawn with replacement by ``weights``; repeatable
    from ``seed``, so a resumed run trains the epochs it would have."""
    import random

    return random.Random(seed).choices(range(len(weights)), weights=list(weights), k=n)

In [ ]:
import warnings

import torch
import transformers
from huggingface_hub import HfApi, hf_hub_download, snapshot_download

import distill
import ftkit

transformers.logging.set_verbosity_error()
warnings.filterwarnings("ignore", module="transformers")
ftkit.fast_cuda()
TOKEN = os.environ["HF_TOKEN"]
api = HfApi(token=TOKEN)

teacher = json.loads(Path(hf_hub_download(FLEX_REPO, "teacher-smoke.json" if SMOKE else "teacher.json",
                                          token=TOKEN)).read_text("utf-8"))
TEACHER = teacher["run"]
LABELS = f"{PREFIX}/{'pseudo-smoke' if SMOKE else 'pseudo'}/{TEACHER.replace('/', '--')}"
if SMOKE:
    LIMIT = LIMIT or 500
OUT = FT / "out" / LABELS
(OUT / "shards").mkdir(parents=True, exist_ok=True)
print("teacher:", TEACHER, f"(val {teacher['val_wer']:.2f}, gold {teacher['gold_wer']:.2f}) | labels:", LABELS)

with ftkit.timed("downloading the corpus"):
    CORPUS = Path(snapshot_download(DATASET_REPO, repo_type="dataset", token=TOKEN,
                                    allow_patterns=[f"{PREFIX}/clips.jsonl", f"{PREFIX}/episodes/*"]))
clips = [json.loads(line) for line in (CORPUS / PREFIX / "clips.jsonl").read_text("utf-8").splitlines() if line]
unmeasured = sum(c.get("overlap_share") is None for c in clips)
if unmeasured:
    print(f"{unmeasured} of {len(clips)} clips have no overlap measured: run 04_PreDistill.ipynb's overlap "
          "pass first, or the overlap rule keeps them all")
if LIMIT:
    clips = clips[:LIMIT]
with ftkit.timed("reading the recordings into RAM"):
    store = ftkit.AudioStore(CORPUS, [c["episode_id"] for c in clips], folder=f"{PREFIX}/episodes")
print(f"audio in RAM: {store.gib:.1f} GiB")
with ftkit.timed("downloading the teacher"):
    TEACHER_DIR = Path(snapshot_download(FLEX_REPO, allow_patterns=[f"{TEACHER}/best/*"], token=TOKEN)) / TEACHER / "best"
sys.path.insert(0, str(TEACHER_DIR))
from indic_transcribe import MODES, IndicTranscribe  # noqa: E402

asr = IndicTranscribe.from_pretrained(str(TEACHER_DIR), device="cuda", dtype=torch.float32)
model, featurize, tk = asr.model, asr.fe, asr.tokenizer
model.eval()
itn, romanized = MODES[MODE]
PROMPT = tk.encode_prompt(LANG, itn=itn, romanized=romanized)
EOS, PAD = tk.eos_id, tk.pad_id
print(f"{len(clips)} clips, {sum(ftkit.duration(c) for c in clips) / 3600:.1f} h, from "
      f"{len({c['source_id'] for c in clips})} sources")

## The rate train's labels span, in the teacher's tokens

The filter's rate rule keeps a clip whose tokens per second fall inside the range the verified
train labels span when the teacher's tokenizer writes them, as `03a_Flex_Train.ipynb` encodes its
targets. The labels are the export the teacher was trained on.

In [ ]:
DEV_DIGITS = str.maketrans("०१२३४५६७८९", "0123456789")


def normalize(text: str) -> str:
    """03a_Flex_Train.ipynb's target normalisation: the characters the tokenizer lacks, mapped onto
    ones fold.py scores as identical."""
    return (text.replace("।", ".").translate(DEV_DIGITS).replace("—", "-")
            .replace("‍", "").replace("‌", ""))


with ftkit.timed("reading train's labels"):
    labelled = ftkit.download_dataset()
    splits = ftkit.load_splits(labelled)
exported = json.loads((labelled / "training" / "manifest.json").read_text())["exported_at"]
if exported != teacher["dataset_export"]:
    print(f"the labelled export on the hub ({exported}) is not the one the teacher trained on "
          f"({teacher['dataset_export']}); the rate range below is this export's")
rates = sorted(len(tk.multi.encode(normalize(r["text"]), out_type=int)) / ftkit.duration(r)
               for r in splits["train"])
RATE_RANGE = (rates[0], rates[-1])
print(f"train: {len(rates)} clips, {RATE_RANGE[0]:.2f} to {RATE_RANGE[1]:.2f} teacher tokens/s "
      f"(median {rates[len(rates) // 2]:.2f})")

## Decode, shard by shard

Greedy, fp32 weights under bf16 autocast (the decoding gold is scored with). Each clip keeps its
text, its token count and its mean log-prob over the tokens up to and including EOS, and whether
its output loops. A shard is uploaded as soon as it is decoded.

In [ ]:
def transcribe_scored(rows):
    """Greedy decode of a batch, with each clip's mean log-prob and token count."""
    audio = [store.clip_f32(r) for r in rows]
    wav = torch.zeros(len(rows), ftkit.pad_len(max(len(a) for a in audio), PAD_TO_S))
    for i, a in enumerate(audio):
        wav[i, :len(a)] = a
    lens = torch.tensor([len(a) for a in audio], device="cuda")
    feats, flens = featurize(wav.cuda(), lens)  # fp32: it disables autocast itself
    mask = (torch.arange(feats.size(2), device="cuda")[None, :] < flens[:, None]).long()
    prompt = torch.tensor([PROMPT] * len(rows), device="cuda")
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
        out = model.generate(input_features=feats, attention_mask=mask, decoder_input_ids=prompt,
                             do_sample=False, num_beams=1, eos_token_id=EOS, pad_token_id=PAD,
                             max_new_tokens=MAX_NEW_TOKENS, output_scores=True,
                             return_dict_in_generate=True)
    logprobs = model.compute_transition_scores(out.sequences, out.scores, normalize_logits=True).float()
    new = out.sequences[:, -logprobs.shape[1]:]
    eos = new == EOS
    counted = (eos.cumsum(1) - eos.long()) == 0  # every step up to and including the first EOS
    mean_lp = (logprobs.nan_to_num(0.0) * counted).sum(1) / counted.sum(1).clamp(min=1)
    n_tokens = counted.sum(1) - eos.any(1).long()
    texts = [tk.decode(tk.strip_prompt_and_trim(o.tolist(), PROMPT)) for o in out.sequences]
    return [{"text": t.strip(), "n_tokens": int(n), "mean_logprob": round(float(lp), 5),
             "looped": ftkit.is_loop(t)}
            for t, n, lp in zip(texts, n_tokens.tolist(), mean_lp.tolist(), strict=True)]


def shard_names():
    prefix = f"{LABELS}/shards/"
    return sorted(f for f in api.list_repo_files(DATASET_REPO, repo_type="dataset") if f.startswith(prefix))


done = set()
for name in shard_names():
    for line in Path(hf_hub_download(DATASET_REPO, name, repo_type="dataset", token=TOKEN)).read_text("utf-8").splitlines():
        done.add(json.loads(line)["segment_id"])
todo = [c for c in clips if c["segment_id"] not in done]
print(f"{len(done)} clips already labelled, {len(todo)} to decode "
      f"({sum(ftkit.duration(c) for c in todo) / 3600:.1f} h)")
first = len(shard_names())
for k in range(0, len(todo), SHARD_CLIPS):
    part, number = todo[k:k + SHARD_CLIPS], first + k // SHARD_CLIPS + 1
    rows, compute = [], 0.0
    with ftkit.timed(f"shard {number}: decoding {len(part)} clips"):
        for batch in ftkit.bucket_batches(part, budget_s=DECODE_BUDGET_S, max_items=DECODE_ITEMS,
                                          pad_to_s=PAD_TO_S, shuffle=False):
            chunk = [part[i] for i in batch]
            for c, r in zip(chunk, transcribe_scored(chunk), strict=True):
                rows.append({"segment_id": c["segment_id"], "source_id": c["source_id"],
                             "duration": round(ftkit.duration(c), 3), **r, "teacher": TEACHER})
    looped = sum(r["looped"] for r in rows)
    print(f"shard {number}: {looped} looped, mean log-prob "
          f"{sum(r['mean_logprob'] for r in rows) / len(rows):.3f}")
    local = OUT / "shards" / f"part-{number:05d}.jsonl"
    local.write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in rows), "utf-8")
    with ftkit.timed(f"shard {number}: uploading"):
        api.upload_file(path_or_fileobj=str(local), path_in_repo=f"{LABELS}/shards/{local.name}",
                        repo_id=DATASET_REPO, repo_type="dataset",
                        commit_message=f"{LABELS}: shard {number} ({len(rows)} clips)")

## Filter, and what is left to train on

Every shard is read back from the repo, so this cell works on whatever has been labelled so far,
and it is the only cell to run again after changing `MAX_OVERLAP_SHARE` or `DROP_FRACTION`. Each
label is joined with its clip's times, channel and overlap, so `labels.jsonl` is all a student
notebook needs beside the audio. The report says what each rule dropped, and what was kept per
channel, per overlap bucket and per share of English words.

In [ ]:
by_id = {c["segment_id"]: c for c in clips}
rows = []
for name in shard_names():
    path = Path(hf_hub_download(DATASET_REPO, name, repo_type="dataset", token=TOKEN))
    for line in path.read_text("utf-8").splitlines():
        r = json.loads(line)
        if (c := by_id.get(r["segment_id"])) is not None:  # a clip outside LIMIT is left out
            rows.append({**r, **{k: c.get(k) for k in ("episode_id", "channel", "start_time", "end_time",
                                                       "overlap_share")}})
kept, report = distill.filter_pseudo(rows, tokens_per_s=RATE_RANGE, drop_fraction=DROP_FRACTION,
                                     max_overlap_share=MAX_OVERLAP_SHARE)
kept_ids = {r["segment_id"] for r in kept}


def tally(key):
    # clips, hours and kept hours per value of `key(row)`, and the teacher's mean confidence
    out = {}
    for r in rows:
        b = out.setdefault(key(r), {"clips": 0, "hours": 0.0, "kept_hours": 0.0, "lp": 0.0})
        b["clips"] += 1
        b["hours"] += r["duration"] / 3600
        b["kept_hours"] += r["duration"] / 3600 * (r["segment_id"] in kept_ids)
        b["lp"] += r["mean_logprob"]
    return {str(k): {"clips": v["clips"], "hours": round(v["hours"], 2), "kept_hours": round(v["kept_hours"], 2),
                     "mean_logprob": round(v["lp"] / v["clips"], 4)}
            for k, v in sorted(out.items(), key=lambda kv: str(kv[0]))}


report["by_channel"] = tally(lambda r: r["channel"])
report["by_overlap"] = tally(lambda r: distill.overlap_bucket(r["overlap_share"]))
report["by_english_share"] = tally(lambda r: distill.mixing_bucket(distill.latin_share(r["text"])))
report["teacher"], report["rate_range"] = TEACHER, list(RATE_RANGE)
print(json.dumps({k: v for k, v in report.items() if not k.startswith("by_")}, indent=1))
for name in ("by_channel", "by_overlap", "by_english_share"):
    print(f"\n{name[3:]:<40} {'clips':>7} {'hours':>7} {'kept h':>7} {'kept':>6} {'log-prob':>9}")
    for value, b in report[name].items():
        print(f"  {value:<38} {b['clips']:>7} {b['hours']:>7.1f} {b['kept_hours']:>7.1f} "
              f"{b['kept_hours'] / max(b['hours'], 1e-9):>6.0%} {b['mean_logprob']:>9.3f}")
(OUT / "labels.jsonl").write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in kept), "utf-8")
(OUT / "report.json").write_text(json.dumps(report, indent=1), "utf-8")
with ftkit.timed("uploading the filtered labels and the report"):
    for f in ("labels.jsonl", "report.json"):
        api.upload_file(path_or_fileobj=str(OUT / f), path_in_repo=f"{LABELS}/{f}", repo_id=DATASET_REPO,
                        repo_type="dataset", commit_message=f"{LABELS}: filtered, {report['kept']} clips kept")